## Correlation with base inconsistency
In this section I will look at the connection between the inconsistencies in the base forecasts and the reconciled error metrics. The incosistency is calculated as the difference between the monthly forecasts and the sum of daily forecasts.

In [ ]:
import os, pathlib
while pathlib.Path.cwd().name != 'forecast_research':
    os.chdir('..') # Run from repo root

import pandas as pd
import plotly.express as px
import python.utils as utils
import python.plots as plots
from python.time_series import BASE_FORECASTS, RECONCILED_FORECASTS, TimeSeriesType
from python.aggregation import MONTHS, FORTNIGHTS, WEEKS, WEEKDAYS, WEEKENDS, DAYS
from python.temporal_hierarchy import TemporalHierarchy, MinimalTemporalHierarchy, SemanticTemporalHierarchy
from python.error_metrics import ErrorMetrics, MergedErrorMetrics, MetricsIndex, LoadedErrorMetrics, ErrorDifference, FilteredErrorMetrics, ClonedErrorMetrics, CustomMetrics
from python.error_metrics import MetricsIndex as MI # Alias
from python.forecast_reconciliation import ForecastReconciliation

In [ ]:
em = LoadedErrorMetrics('metrics/base_hier_errors.csv')
inc = pd.read_csv('metrics/base_hier_inconsistency.csv', index_col=0)

sth = SemanticTemporalHierarchy()
mth = MinimalTemporalHierarchy()
wth = TemporalHierarchy([MONTHS, WEEKS, DAYS], 'Minimal+Weeks')
wweth = TemporalHierarchy([MONTHS, WEEKS, WEEKDAYS, WEEKENDS, DAYS], 'Weekends+Weekdays')
wfth = TemporalHierarchy([MONTHS, FORTNIGHTS, WEEKS, DAYS], 'Weeks+Biweeks')

error_metric = 'nrmse'
em = FilteredErrorMetrics(em, MI(error_metric='nrmse'))

diffs = MergedErrorMetrics([
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=sth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wweth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=mth)),
    ErrorDifference(em, MI(forecast_type=BASE_FORECASTS, hierarchy=mth), MI(forecast_type=RECONCILED_FORECASTS, hierarchy=wfth)),
])

nrmse_em, nrmse_diffs = em, diffs

### Differences as function of base inconsistency
Here we visually how the base forecast inconsistency might affect the hierarchy's reconciliation performance measured by the difference made to the errors at each granularity independently

In [ ]:
for h in (sth, mth, wth, wweth, wfth):
    plots.plot_error_over_inconsistency(nrmse_diffs,
                                        MI(hierarchy=f'{h} - Minimal', granularity=DAYS),
                                        inc,
                                        outlier_threshold=0.99).show()

Looking at the differences made to daily granularity error, in all hierarchies some pattern start to emerge which show a greater number of negative changes (improvements) with negative inconsistency (meaning the sum of Days was greater than the monthly forecast), while there are more positive changes (decline in accuracy) when this inconsistency is positive.

In [ ]:
for h in (sth, mth, wth, wweth, wfth):
    plots.plot_error_over_inconsistency(nrmse_diffs,
                                        MI(hierarchy=f'{h} - Minimal', granularity=MONTHS),
                                        inc,
                                        outlier_threshold=0.99).show()

At monthly granularity, we can see the opposite pattern emerge (as if the shape from the daily granularity plot was flipped upside down): when the inconsistency is positive (meaning monthly forecast was greater than the sum of daily), we tend to see more improvements in the error metrics.

### Errors as function of base inconsistency
Here I will plot very similar scatter plots, but not with the error diffs, but with the raw errors. First I will look at the base forecast errors.

In [ ]:
for g in (DAYS, MONTHS):
    plots.plot_error_over_inconsistency(nrmse_em,
                                        MI(forecast_type=BASE_FORECASTS, granularity=g),
                                        inc,
                                        outlier_threshold=0.99).show()

It is interesting to see that the biggest errors happen at the smallest (absolute value) inconsistencies. But otherwise, no other recognizable pattern emerges.

Next we look at the same scatter plots but with the reconciled forecast errors on the Y axis, first at daily granularity.

In [ ]:
for h in (sth, mth, wth, wweth, wfth):
    plots.plot_error_over_inconsistency(nrmse_em,
                                        MI(forecast_type=RECONCILED_FORECASTS, hierarchy=h, granularity=DAYS),
                                        inc,
                                        outlier_threshold=0.99).show()

There are no significant differences visually between the plots. Now we look at monthly granularity.

In [ ]:
for h in (sth, mth, wth, wweth, wfth):
    plots.plot_error_over_inconsistency(nrmse_em,
                                        MI(forecast_type=RECONCILED_FORECASTS, hierarchy=h, granularity=MONTHS),
                                        inc,
                                        outlier_threshold=0.99).show()

The shape is different from the daily granularity case, as expected, but over the different hierarchies, there is still no visible difference.

### Conclusions
No interesting finding came out of this sections